In [29]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import sys
import os

## API
**Original dataset:** https://graph.openaire.eu/docs/

**API:** https://api.openaire.eu/graph/swagger-ui/index.html#/Research%20products/search

We start by analyzing the organizations (`organizations`). Since our focus is the *University of Pisa*, this filter is the most direct, so we retrieve all entries containing "Pisa".

In [30]:
url = "https://api.openaire.eu/graph/v1/organizations?search=Pisa&page=1&pageSize=100&sortBy=relevance%20DESC"
headers = {"accept": "application/json"}

try:
    response = requests.get(url, headers=headers, timeout=30)
    if response.ok:
        data_Pisa = response.json()
        print("Response loaded in 'data_Pisa'.")
        print(f"Status: {response.status_code}, Records: {len(data_Pisa.get('results', []))}")
    else:
        data_Pisa = None
        print(f"Error: {response.status_code} - {response.reason}")
except requests.exceptions.RequestException as e:
    data_Pisa = None
    print(f"Network error: {e}")

print("Data keys:", data_Pisa.keys())


Response loaded in 'data_Pisa'.
Status: 200, Records: 50
Data keys: dict_keys(['header', 'results'])


In [31]:
# Print unique structure of 'results' 
if "results" in data_Pisa:
    # Get the keys from the first result (assuming all have same structure)
    if data_Pisa["results"]:
        print("Base structure of results:")
        for key in data_Pisa["results"][0].keys():
            print(f"  {key}")
        
        # Verify all results have the same structure
        first_keys = set(data_Pisa["results"][0].keys())
        all_same = all(set(result.keys()) == first_keys for result in data_Pisa["results"])
        print(f"\nAll results have the same structure: {all_same}")
        
        if not all_same:
            print("Different structures found:")
            key_structures = [tuple(sorted(result.keys())) for result in data_Pisa["results"]]
            from collections import Counter
            counter = Counter(key_structures)
            for keys, count in counter.items():
                print(f"Structure: {list(keys)} - Count: {count}")

Base structure of results:
  legalShortName
  legalName
  websiteUrl
  alternativeNames
  country
  id
  pids

All results have the same structure: True


In [32]:
def analyze_countries(data, data_key="results"):
    """
    Analyze unique country values in the dataset.
    
    Args:
        data (dict): The dataset containing results
        data_key (str): The key containing the results array (default: "results")
    """
    if data_key in data and data[data_key]:
        countries = {str(entry.get("country")) for entry in data[data_key]}
        print("Unique country values:")
        for country in sorted(countries):
            print(f"  {country}")
        print(f"\nTotal unique countries: {len(countries)}")
        
        # Show the structure of country objects
        print("\nCountry structure (first example):")
        first_country = data[data_key][0].get("country")
        if isinstance(first_country, dict):
            for key, value in first_country.items():
                print(f"  {key}: {value}")
    else:
        print("No results found to analyze countries.")

# Call the function
analyze_countries(data_Pisa)

Unique country values:
  {'code': 'CH', 'label': 'Switzerland'}
  {'code': 'FI', 'label': 'Finland'}
  {'code': 'FR', 'label': 'France'}
  {'code': 'IT', 'label': 'Italy'}
  {'code': 'UNKNOWN', 'label': 'Unknown'}
  {'code': 'US', 'label': 'United States'}

Total unique countries: 6

Country structure (first example):
  code: IT
  label: Italy


In [33]:
# Filter for IT and UNKNOWN countries and modify results directly
if "results" in data_Pisa and data_Pisa["results"]:
    original_count = len(data_Pisa["results"])
    
    # Filter and modify results directly
    data_Pisa["results"] = [
        entry for entry in data_Pisa["results"] 
        if entry.get("country", {}).get("code") in ["IT", "UNKNOWN"]
    ]
    
    print(f"Original total: {original_count}")
    print(f"After filtering for IT and UNKNOWN: {len(data_Pisa['results'])}")
    print(f"Removed: {original_count - len(data_Pisa['results'])}")
    
    # Show breakdown by country
    country_counts = {}
    for entry in data_Pisa["results"]:
        country_code = entry.get("country", {}).get("code")
        country_counts[country_code] = country_counts.get(country_code, 0) + 1
    
    print("\nBreakdown:")
    for code, count in country_counts.items():
        print(f"  {code}: {count}")
else:
    print("No results found to analyze.")

Original total: 50
After filtering for IT and UNKNOWN: 37
Removed: 13

Breakdown:
  IT: 22
  UNKNOWN: 15


In [34]:
analyze_countries(data_Pisa, "results")

Unique country values:
  {'code': 'IT', 'label': 'Italy'}
  {'code': 'UNKNOWN', 'label': 'Unknown'}

Total unique countries: 2

Country structure (first example):
  code: IT
  label: Italy


In [35]:
# Print legalShortName and legalName for entries with country UNKNOWN
if "results" in data_Pisa and data_Pisa["results"]:
    unknown_entries = [
        entry for entry in data_Pisa["results"] 
        if entry.get("country", {}).get("code") == "UNKNOWN"
    ]
    
    print(f"Entries with UNKNOWN country: {len(unknown_entries)}")
    print("\nlegalShortName | legalName")
    print("-" * 50)
    
    for i, entry in enumerate(unknown_entries, 1):
        legal_short = entry.get("legalShortName", "Not available")
        legal_name = entry.get("legalName", "Not available")
        print(f"{i}. {legal_short} | {legal_name}")
else:
    print("No results found.")

Entries with UNKNOWN country: 15

legalShortName | legalName
--------------------------------------------------
1. Scuola Normale Superiore di Pisa | Scuola Normale Superiore di Pisa
2. Università degli Studi di Pisa - Dipartimento die Matematica | Università degli Studi di Pisa - Dipartimento die Matematica
3. University of Toronto Università degli Studi di Pisa Universität Wien | University of Toronto Università degli Studi di Pisa Universität Wien
4. INGV - Pisa | INGV - Pisa
5. Universiteit Utrecht, Faculteit Bètawetenschappen, Departement Informatica, Netherlands Research School for Information and Knowledge Systems (SIKS) | Universiteit Utrecht, Faculteit Bètawetenschappen, Departement Informatica, Netherlands Research School for Information and Knowledge Systems (SIKS)
6. Universiteit Twente, Faculty of Behavioural, Management and Social sciences (BMS), Onderzoeksmethodologie Meetmethoden en Data-Analyse (OMD) | Universiteit Twente, Faculty of Behavioural, Management and Social 

In [36]:
# Create a data structure with all unique legalName values
if "results" in data_Pisa and data_Pisa["results"]:
    unique_legal_names = {entry.get("legalName") for entry in data_Pisa["results"] if entry.get("legalName")}
    
    print(f"Total unique legalName values: {len(unique_legal_names)}")
    print("\nUnique legalName values:")
    for i, legal_name in enumerate(sorted(unique_legal_names), 1):
        print(f"{i}. {legal_name}")
    
    # Convert to list if you need it as a list
    unique_legal_names_list = sorted(list(unique_legal_names))
    print(f"\nData structure created: unique_legal_names_list with {len(unique_legal_names_list)} entries")
else:
    print("No results found.")

Total unique legalName values: 35

Unique legalName values:
1. Associazione Italiana Persone Down Onlus Pisa
2. Associazione nazionale insegnanti scienze naturali Pisa
3. Azienda Ospedaliero Universitaria Pisana
4. British School Pisa srl
5. COMUNE DI PISA
6. Consorzio Pisa Ricerche
7. Department of Chemistry and Industrial Chemistry - Università di Pisa
8. Department of Civilization and forms of knowledge - University of Pisa
9. Dipartimento di Fisica Università degli Studi di Pisa
10. Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa
11. Dipartimento di Studi Italianistici - Università di Pisa
12. Dipartimento di scienze economiche - Università di Pisa
13. Expertisecentrum Nederlands
14. Fondazione Teatro di Pisa
15. INGV - Pisa
16. IPSAR G. MATTEOTTI PISA
17. KBA Nijmegen
18. National Institute for Nuclear Physics
19. PROVINCIA DI PISA
20. Radboud Universiteit Nijmegen, Faculteit der Sociale Wetenschappen, Behavioural Science Institute - BSI
21. Radboud Universitei

In [37]:
# Print unique legalName values in copy-paste format for array creation
if "results" in data_Pisa and data_Pisa["results"]:
    unique_legal_names = {entry.get("legalName") for entry in data_Pisa["results"] if entry.get("legalName")}
    
    print(f"Total unique legalName values: {len(unique_legal_names)}")
    print("\n# Copy-paste ready array:")
    print("legal_names = [")
    for legal_name in sorted(unique_legal_names):
        print(f'    "{legal_name}",')
    print("]")
else:
    print("No results found.")

Total unique legalName values: 35

# Copy-paste ready array:
legal_names = [
    "Associazione Italiana Persone Down Onlus Pisa",
    "Associazione nazionale insegnanti scienze naturali Pisa",
    "Azienda Ospedaliero Universitaria Pisana",
    "British School Pisa srl",
    "COMUNE DI PISA",
    "Consorzio Pisa Ricerche",
    "Department of Chemistry and Industrial Chemistry - Università di Pisa",
    "Department of Civilization and forms of knowledge - University of Pisa",
    "Dipartimento di Fisica Università degli Studi di Pisa",
    "Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa",
    "Dipartimento di Studi Italianistici - Università di Pisa",
    "Dipartimento di scienze economiche - Università di Pisa",
    "Expertisecentrum Nederlands",
    "Fondazione Teatro di Pisa",
    "INGV - Pisa",
    "IPSAR G. MATTEOTTI PISA",
    "KBA Nijmegen",
    "National Institute for Nuclear Physics",
    "PROVINCIA DI PISA",
    "Radboud Universiteit Nijmegen, Faculteit de

In [38]:
legal_names = [
    "Azienda Ospedaliero Universitaria Pisana",
    "Department of Chemistry and Industrial Chemistry - Università di Pisa",
    "Department of Civilization and forms of knowledge - University of Pisa",
    "Dipartimento di Fisica Università degli Studi di Pisa",
    "Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa",
    "Dipartimento di Studi Italianistici - Università di Pisa",
    "Dipartimento di scienze economiche - Università di Pisa",
    "Research Center E. Piaggio - University of Pisa",
    "University of Pisa",
    "University of Pisa, Department of Clinical and Experimental Medecine, Division of Pharmacology, Pisa, Italy",
    "University of Pisa, Faculty of Medicine, Department of Translational Research and of New Surgical and Medical Technologies",
    "Università degli Studi di Pisa - Dipartimento die Matematica",
]

In [39]:
# Filter results to create new object with only entries in legal_names
if "results" in data_Pisa and data_Pisa["results"]:
    # Create filtered dataset
    filtered_data_Pisa = {
        "results": [
            entry for entry in data_Pisa["results"] 
            if entry.get("legalName") in legal_names
        ]
    }
    
    # Copy other keys from original data if they exist
    for key in data_Pisa.keys():
        if key != "results":
            filtered_data_Pisa[key] = data_Pisa[key]
    
    print(f"Original total: {len(data_Pisa['results'])}")
    print(f"After filtering with legal_names: {len(filtered_data_Pisa['results'])}")
    print(f"Removed: {len(data_Pisa['results']) - len(filtered_data_Pisa['results'])}")
    
    # Show which legal names were found
    found_names = {entry.get("legalName") for entry in filtered_data_Pisa["results"]}
    missing_names = set(legal_names) - found_names
    
    print(f"\nFound legal names: {len(found_names)}")
    print(f"Missing legal names: {len(missing_names)}")
    
    if missing_names:
        print("\nMissing names:")
        for name in sorted(missing_names):
            print(f"  - {name}")
else:
    print("No results found to filter.")

Original total: 37
After filtering with legal_names: 11
Removed: 26

Found legal names: 11
Missing legal names: 1

Missing names:
  - University of Pisa, Faculty of Medicine, Department of Translational Research and of New Surgical and Medical Technologies


In [40]:
# Create output directory if it doesn't exist
output_dir = "./output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save only the results array to JSON file
output_file = os.path.join(output_dir, "pisa_organizations.json")

if 'filtered_data_Pisa' in locals() and 'results' in filtered_data_Pisa:
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(filtered_data_Pisa['results'], f, ensure_ascii=False, indent=2)
    
    print(f"Results data saved to: {output_file}")
    print(f"File contains {len(filtered_data_Pisa['results'])} organizations")
else:
    print("No filtered results found to save.")

Results data saved to: ./output/pisa_organizations.json
File contains 11 organizations


In [41]:
import requests
import json
import os
import time
from typing import List, Dict, Set
import hashlib

class OrganizationResearchProductsFetcher:
    def __init__(self, output_dir="research_products_data"):
        """
        Inizializza il fetcher per recuperare i research products delle organizzazioni
        
        Args:
            output_dir: Directory dove salvare i dati
        """
        self.api_url = "https://api.openaire.eu/graph/v2/researchProducts"
        self.output_dir = output_dir
        self.headers = {"accept": "application/json"}
        
        # Crea directory se non esiste
        os.makedirs(output_dir, exist_ok=True)
        
        # Set per tracciare ID già processati (evita duplicati)
        self.processed_ids = set()
        
    def fetch_all_research_products(self, organizations: List[Dict]) -> Dict:
        """
        Recupera TUTTI i research products per una lista di organizzazioni
        
        Args:
            organizations: Lista di dizionari con struttura dell'organizzazione
            
        Returns:
            Dizionario con tutti i research products unici
        """
        all_products = {}  # Usa dizionario per evitare duplicati automaticamente
        
        for org in organizations:
            org_id = org.get("id")
            org_name = org.get("legalShortName", org.get("legalName", "Unknown"))
            
            if not org_id:
                print(f"⚠️  Organizzazione senza ID: {org_name}")
                continue
                
            print(f"\n{'='*60}")
            print(f"Processando: {org_name}")
            print(f"ID: {org_id}")
            print('='*60)
            
            # Recupera prodotti per questa organizzazione
            products = self._fetch_org_products_cursor(org_id)  # Usa versione cursor
            
            # Aggiungi al set globale (evita duplicati usando ID come chiave)
            for product in products:
                product_id = product.get("id")
                if product_id and product_id not in all_products:
                    # Aggiungi info sull'organizzazione al prodotto
                    if "affiliated_organizations" not in product:
                        product["affiliated_organizations"] = []
                    product["affiliated_organizations"].append({
                        "id": org_id,
                        "name": org_name
                    })
                    all_products[product_id] = product
                elif product_id in all_products:
                    # Se il prodotto esiste già, aggiungi solo l'organizzazione
                    if "affiliated_organizations" not in all_products[product_id]:
                        all_products[product_id]["affiliated_organizations"] = []
                    all_products[product_id]["affiliated_organizations"].append({
                        "id": org_id,
                        "name": org_name
                    })
            
            print(f"✓ Trovati {len(products)} prodotti per {org_name}")
            
            # Rate limiting
            time.sleep(0.5)
        
        return all_products
    
    def _fetch_org_products_cursor(self, org_id: str) -> List[Dict]:
        """
        Recupera tutti i prodotti usando CURSOR-BASED pagination (per superare limite 10k)
        
        Args:
            org_id: ID dell'organizzazione
            
        Returns:
            Lista di tutti i research products
        """
        all_products = []
        cursor = "*"  # Cursor iniziale
        page_size = 100
        batch_count = 0
        
        while cursor:
            params = {
                "relOrganizationId": org_id,
                "cursor": cursor,
                "pageSize": page_size,
                "sortBy": "publicationDate DESC"
            }
            
            try:
                response = requests.get(
                    self.api_url, 
                    params=params, 
                    headers=self.headers,
                    timeout=30
                )
                
                if response.status_code != 200:
                    print(f"  ⚠️  Errore {response.status_code} per batch {batch_count}")
                    # Salva progresso parziale se errore
                    if all_products:
                        self._save_partial_progress(org_id, all_products)
                    break
                
                data = response.json()
                results = data.get('results', [])
                
                if not results:
                    break
                
                all_products.extend(results)
                batch_count += 1
                
                # Info progresso
                total = data['header'].get('numFound', 'Unknown')
                print(f"  Batch {batch_count}: recuperati {len(all_products)}/{total} prodotti")
                
                # Ottieni il prossimo cursor
                cursor = data['header'].get('nextCursor')
                
                # Se non c'è nextCursor, abbiamo finito
                if not cursor:
                    break
                
                # Rate limiting più aggressivo per grandi dataset
                if len(all_products) > 5000:
                    time.sleep(1)  # Pausa più lunga dopo 5000 record
                else:
                    time.sleep(0.3)
                    
            except requests.exceptions.RequestException as e:
                print(f"  ⚠️  Errore di rete: {e}")
                # Salva progresso parziale
                if all_products:
                    self._save_partial_progress(org_id, all_products)
                break
            except json.JSONDecodeError as e:
                print(f"  ⚠️  Errore parsing JSON: {e}")
                if all_products:
                    self._save_partial_progress(org_id, all_products)
                break
            except Exception as e:
                print(f"  ⚠️  Errore inaspettato: {e}")
                if all_products:
                    self._save_partial_progress(org_id, all_products)
                break
        
        return all_products
    
    def _fetch_org_products_traditional(self, org_id: str) -> List[Dict]:
        """
        FALLBACK: Metodo tradizionale con paginazione standard (limite 10k)
        Usato se il cursor-based fallisce
        """
        all_products = []
        page = 1
        page_size = 100
        max_pages = 100  # Limite di sicurezza (10k risultati)
        
        while page <= max_pages:
            params = {
                "relOrganizationId": org_id,
                "page": page,
                "pageSize": page_size,
                "sortBy": "publicationDate DESC"
            }
            
            try:
                response = requests.get(
                    self.api_url, 
                    params=params, 
                    headers=self.headers,
                    timeout=30
                )
                
                if response.status_code != 200:
                    print(f"  ⚠️  Errore {response.status_code} per pagina {page}")
                    break
                
                data = response.json()
                results = data.get('results', [])
                
                if not results:
                    break
                
                all_products.extend(results)
                
                # Info progresso
                total = data['header']['numFound']
                print(f"  Pagina {page}: recuperati {len(all_products)}/{total} prodotti")
                
                # Avviso se ci sono più di 10k risultati
                if total > 10000 and page == 1:
                    print(f"  ⚠️  ATTENZIONE: Ci sono {total} prodotti ma il limite è 10.000")
                    print(f"     Usa cursor-based pagination per recuperarli tutti!")
                
                # Controlla se ci sono altre pagine
                if len(all_products) >= total or len(all_products) >= 10000:
                    break
                    
                page += 1
                
            except Exception as e:
                print(f"  ⚠️  Errore: {e}")
                break
        
        return all_products
    
    def _save_partial_progress(self, org_id: str, products: List[Dict]):
        """
        Salva progresso parziale in caso di errore
        """
        partial_file = os.path.join(
            self.output_dir, 
            f"partial_{org_id.replace(':', '_')}.json"
        )
        with open(partial_file, 'w', encoding='utf-8') as f:
            json.dump(products, f, indent=2, ensure_ascii=False)
        print(f"  💾 Salvato progresso parziale in: {partial_file}")
    
    def save_products(self, products: Dict, filename_prefix="research_products"):
        """
        Salva i research products in file JSON locali
        
        Args:
            products: Dizionario dei research products
            filename_prefix: Prefisso per il nome del file
        """
        # Converti dizionario in lista per il salvataggio
        products_list = list(products.values())
        
        # File principale con tutti i dati
        main_file = os.path.join(self.output_dir, f"{filename_prefix}_complete.json")
        with open(main_file, 'w', encoding='utf-8') as f:
            json.dump(products_list, f, indent=2, ensure_ascii=False)
        
        print(f"\n✓ Salvati {len(products_list)} prodotti unici in: {main_file}")
        
        # Salva anche versione compatta (solo info essenziali)
        compact_file = os.path.join(self.output_dir, f"{filename_prefix}_compact.json")
        compact_products = []
        
        for product in products_list:
            compact = {
                "id": product.get("id"),
                "mainTitle": product.get("mainTitle"),
                "publicationDate": product.get("publicationDate"),
                "type": product.get("type"),
                "doi": None,
                "affiliated_organizations": product.get("affiliated_organizations", [])
            }
            
            # Estrai DOI se disponibile
            if 'pid' in product:
                for pid in product['pid']:
                    if pid.get('scheme') == 'doi':
                        compact["doi"] = pid.get('value')
                        break
            
            compact_products.append(compact)
        
        with open(compact_file, 'w', encoding='utf-8') as f:
            json.dump(compact_products, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Salvata versione compatta in: {compact_file}")
        
        # Salva statistiche
        self._save_statistics(products_list)
    
    def _save_statistics(self, products: List[Dict]):
        """
        Calcola e salva statistiche sui research products
        """
        stats = {
            "total_products": len(products),
            "by_type": {},
            "by_year": {},
            "with_doi": 0,
            "organizations_involved": set()
        }
        
        for product in products:
            # Per tipo
            ptype = product.get("type", "unknown")
            stats["by_type"][ptype] = stats["by_type"].get(ptype, 0) + 1
            
            # Per anno
            if product.get("publicationDate"):
                year = product["publicationDate"][:4]
                stats["by_year"][year] = stats["by_year"].get(year, 0) + 1
            
            # Con DOI
            if 'pid' in product:
                for pid in product['pid']:
                    if pid.get('scheme') == 'doi':
                        stats["with_doi"] += 1
                        break
            
            # Organizzazioni
            for org in product.get("affiliated_organizations", []):
                stats["organizations_involved"].add(org["name"])
        
        # Converti set in lista per JSON
        stats["organizations_involved"] = list(stats["organizations_involved"])
        
        stats_file = os.path.join(self.output_dir, "statistics.json")
        with open(stats_file, 'w', encoding='utf-8') as f:
            json.dump(stats, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Statistiche salvate in: {stats_file}")
        
        # Stampa summary
        print("\n📊 STATISTICHE:")
        print(f"  - Prodotti totali unici: {stats['total_products']}")
        print(f"  - Con DOI: {stats['with_doi']}")
        print(f"  - Organizzazioni coinvolte: {len(stats['organizations_involved'])}")
        print(f"  - Tipi di prodotto: {stats['by_type']}")


In [42]:
with open("./output/pisa_organizations.json", 'r') as f:
    organizations = json.load(f)

# Recupera e salva tutti i prodotti
fetcher = OrganizationResearchProductsFetcher()
products = fetcher.fetch_all_research_products(organizations)
fetcher.save_products(products)


Processando: Department of Civilization and forms of knowledge - University of Pisa
ID: openorgs____::c186a10ae4acdf58596bacc91c2cc620
  Batch 1: recuperati 60/60 prodotti
✓ Trovati 60 prodotti per Department of Civilization and forms of knowledge - University of Pisa

Processando: Dipartimento di Chimica e Chimica Industriale
ID: openorgs____::0462274ad38f4d94dbe86d320960850e
  Batch 1: recuperati 49/49 prodotti
✓ Trovati 49 prodotti per Dipartimento di Chimica e Chimica Industriale

Processando: Dipartimento di Studi Italianistici - Università di Pisa
ID: openorgs____::6cac13134787940043685a42cc596d23
✓ Trovati 0 prodotti per Dipartimento di Studi Italianistici - Università di Pisa

Processando: Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa
ID: pending_org_::ab0cf7bec4e87cabe665fa48b5ae9a79
  Batch 1: recuperati 57/57 prodotti
✓ Trovati 57 prodotti per Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa

Processando: Dipartimento di scienze econ